In [1]:
test_suite = {
  "test_suite_name": "Medical Coreference Resolution & Semantic Extraction Test Suite",
  "test_cases": [
    {
      "id": 1,
      "description": "Rule 1 & 2: Basic Medication Pronoun Resolution",
      "input_text": "Patient James Miller was prescribed Metformin for Type 2 diabetes. It is a first-line medication. He must take it twice daily with meals to regulate his blood sugar levels."
    },
    {
      "id": 2,
      "description": "Rule 3: Carrying Antecedent Across Distant Sentences",
      "input_text": "Dr. Sarah Chen admitted Mrs. Higgins on Monday. The patient presented with severe abdominal pain. After a series of tests, she was diagnosed with acute appendicitis. She underwent surgery that evening."
    },
    {
      "id": 3,
      "description": "Rule 4: Canonical Name Resolution (Epithets)",
      "input_text": "Professor Robert Lang is a world-renowned oncologist. The specialist led the clinical trial for the new immunotherapy. The researcher claimed the results were unprecedented for Stage IV lung cancer."
    },
    {
      "id": 4,
      "description": "Abstract 'This/That' Event Resolution",
      "input_text": "The patient suffered a myocardial infarction in 2022. This event caused significant damage to the left ventricle. That complication led to the eventual installation of a pacemaker."
    },
    {
      "id": 5,
      "description": "Split Antecedents (Multiple People)",
      "input_text": "Nurse Kelly and Dr. Varma entered the operating room. They prepared the site for the incision. Both professionals confirmed the patient's identity before beginning the procedure."
    },
    {
      "id": 6,
      "description": "Rule 4: Acronym and Institutional Synonym Resolution",
      "input_text": "The Centers for Disease Control and Prevention (CDC) released new guidelines. The agency recommended updated boosters. The organization emphasized that the virus is still evolving."
    },
    {
      "id": 7,
      "description": "Class-Member/Hypernymy Resolution",
      "input_text": "The patient was treated with Amoxicillin. The antibiotic was administered intravenously. However, the drug caused an allergic reaction characterized by a rash."
    },
    {
      "id": 8,
      "description": "Possessive Pronoun Resolution in Clinical Context",
      "input_text": "Liam Smith visited the dermatology clinic. His psoriasis has worsened over the last month. Its plaques are now appearing on his elbows and knees."
    },
    {
      "id": 9,
      "description": "Cataphora (Pronoun Preceding Entity)",
      "input_text": "Because it was causing a blockage, the surgeon removed the gallbladder. The organ showed signs of chronic cholecystitis during the pathology review."
    },
    {
      "id": 10,
      "description": "Temporal Entity Relative Resolution",
      "input_text": "The clinical trial started in January 2024. Six months later, the first phase was completed. By December, the findings were published in the Lancet."
    },
    {
      "id": 11,
      "description": "Collective Noun Resolution (Medical Teams)",
      "input_text": "The Ethics Committee met on Thursday morning. They reviewed the proposal for the organ transplant. The group eventually granted approval for the surgery."
    },
    {
      "id": 12,
      "description": "Reflexive Pronoun in Biological Processes",
      "input_text": "In autoimmune diseases, the immune system attacks itself. It mistakes healthy cells for foreign invaders. This process leads to chronic inflammation in the joints."
    },
    {
      "id": 13,
      "description": "Device and Component Coreference",
      "input_text": "The MRI machine at St. Jude’s requires maintenance. Its liquid helium levels are low. The technician must refill it before the next scheduled scan."
    },
    {
      "id": 14,
      "description": "Rule 3: Implicit Object Resolution",
      "input_text": "The doctor handed the prescription to the pharmacist. He filled it immediately. The patient then took the bottle and left the pharmacy."
    },
    {
      "id": 15,
      "description": "Anaphoric Reference to Therapy",
      "input_text": "The patient underwent Cognitive Behavioral Therapy. This treatment lasted for twelve sessions. The aforementioned therapy helped reduce his anxiety symptoms significantly."
    },
    {
      "id": 16,
      "description": "Former/Latter in Diagnostic Comparisons",
      "input_text": "Both CT scans and MRIs were performed on the brain. The former showed no hemorrhage, while the latter revealed a small lesion in the frontal lobe."
    },
    {
      "id": 17,
      "description": "Rule 4: Consolidating Varied Descriptions of Disease",
      "input_text": "The patient lives with Amyotrophic Lateral Sclerosis. ALS is a progressive neurodegenerative disease. This condition primarily affects motor neurons in the brain and spinal cord."
    },
    {
      "id": 18,
      "description": "Geopolitical/Location Coreference",
      "input_text": "The Mayo Clinic is located in Rochester. The institution is famous for its collaborative care model. Many international patients travel to the facility for complex surgeries."
    },
    {
      "id": 19,
      "description": "Complex Gender/Role Ambiguity",
      "input_text": "A female surgeon and a male nurse discussed the case. She asked him to check the patient's vitals. He reported that they were stable."
    },
    {
      "id": 20,
      "description": "Negation and Multiple Antecedents",
      "input_text": "Neither Ibuprofen nor Aspirin was effective for the chronic pain. They both caused gastric upset. Therefore, the physician switched the patient to a different class of analgesics."
    }
  ]
}

In [2]:
from text_decomposition_prompt import text_decomposition_prompt
print(text_decomposition_prompt("TEST"))


--Goal--
Given a text, segment it into multiple semantic units, each containing detailed descriptions of specific events or activities.
Perform the following tasks:
--Steps--
1. Provide a summary for each semantic unit while retaining all crucial details relevant to the original context.
2. Extract all entities directly from the original text of each semantic unit, not from the paraphrased. Format each entity name in UPPERCASE. You should extract all entities including times, locations, people, organizations and all kinds of entities.
3. From the entities extracted in Step 2, list all relationships within the semantic unit and the corresponding original context in the form of string separated by comma: "ENTITY_A, RELATION_TYPE, ENTITY_B". The RELATION_TYPE could be a descriptive sentence, while the entities involved in the relationship must come from the entity names extracted in Step 2. Please make sure the string contains three elements representing two entities and the relationship 

In [3]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: d:\Documents\GitHub\NodeRAG\text_decomposition
The root directory is: d:\Documents\GitHub\NodeRAG


In [4]:
#LLM api call
from google import genai

with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text,
    )
    return response.text, response.usage_metadata.total_token_count

In [5]:
responses = {}
from tqdm import tqdm
for tc in test_suite["test_cases"]:
    input_text = tc["input_text"]
    prompt = text_decomposition_prompt(input_text)
    res, _ = call_gemini(prompt)
    responses[input_text] = res

In [6]:
responses

{'Patient James Miller was prescribed Metformin for Type 2 diabetes. It is a first-line medication. He must take it twice daily with meals to regulate his blood sugar levels.': '[\n  {\n    "semantic_unit": "Patient James Miller was prescribed Metformin to treat his Type 2 diabetes.",\n    "entities": [\n      "JAMES MILLER",\n      "METFORMIN",\n      "TYPE 2 DIABETES"\n    ],\n    "relationships": [\n      "JAMES MILLER, was prescribed, METFORMIN",\n      "METFORMIN, for, TYPE 2 DIABETES",\n      "JAMES MILLER, has, TYPE 2 DIABETES"\n    ]\n  },\n  {\n    "semantic_unit": "Metformin is identified as a first-line medication.",\n    "entities": [\n      "METFORMIN",\n      "FIRST-LINE MEDICATION"\n    ],\n    "relationships": [\n      "METFORMIN, is a, FIRST-LINE MEDICATION"\n    ]\n  },\n  {\n    "semantic_unit": "James Miller is instructed to take Metformin twice daily with meals to regulate his blood sugar levels.",\n    "entities": [\n      "JAMES MILLER",\n      "METFORMIN",\n    

In [ ]:
import json
with open('data/prompt_test_result.json', 'w', encoding='utf-8') as f:
    json.dump(responses, f, indent=4, ensure_ascii=False)
with open('data/prompt_test_result.json', 'r', encoding='utf-8') as f:
    loaded_responses = json.load(f)

In [8]:
loaded_responses

{'Patient James Miller was prescribed Metformin for Type 2 diabetes. It is a first-line medication. He must take it twice daily with meals to regulate his blood sugar levels.': '[\n  {\n    "semantic_unit": "Patient James Miller was prescribed Metformin to treat his Type 2 diabetes.",\n    "entities": [\n      "JAMES MILLER",\n      "METFORMIN",\n      "TYPE 2 DIABETES"\n    ],\n    "relationships": [\n      "JAMES MILLER, was prescribed, METFORMIN",\n      "METFORMIN, for, TYPE 2 DIABETES",\n      "JAMES MILLER, has, TYPE 2 DIABETES"\n    ]\n  },\n  {\n    "semantic_unit": "Metformin is identified as a first-line medication.",\n    "entities": [\n      "METFORMIN",\n      "FIRST-LINE MEDICATION"\n    ],\n    "relationships": [\n      "METFORMIN, is a, FIRST-LINE MEDICATION"\n    ]\n  },\n  {\n    "semantic_unit": "James Miller is instructed to take Metformin twice daily with meals to regulate his blood sugar levels.",\n    "entities": [\n      "JAMES MILLER",\n      "METFORMIN",\n    

In [11]:
c = 1
for r in loaded_responses:
    print("Test cases:",c,"/ 20")
    print(r)
    print(loaded_responses[r])
    c+=1
    print("-"*100)

Test cases: 1 / 20
Patient James Miller was prescribed Metformin for Type 2 diabetes. It is a first-line medication. He must take it twice daily with meals to regulate his blood sugar levels.
[
  {
    "semantic_unit": "Patient James Miller was prescribed Metformin to treat his Type 2 diabetes.",
    "entities": [
      "JAMES MILLER",
      "METFORMIN",
      "TYPE 2 DIABETES"
    ],
    "relationships": [
      "JAMES MILLER, was prescribed, METFORMIN",
      "METFORMIN, for, TYPE 2 DIABETES",
      "JAMES MILLER, has, TYPE 2 DIABETES"
    ]
  },
  {
    "semantic_unit": "Metformin is identified as a first-line medication.",
    "entities": [
      "METFORMIN",
      "FIRST-LINE MEDICATION"
    ],
    "relationships": [
      "METFORMIN, is a, FIRST-LINE MEDICATION"
    ]
  },
  {
    "semantic_unit": "James Miller is instructed to take Metformin twice daily with meals to regulate his blood sugar levels.",
    "entities": [
      "JAMES MILLER",
      "METFORMIN",
      "TWICE DAILY"